# Data Wrangling
## Lab 3 — Real Estate Market

**Dataset:** California Housing dataset from `sklearn.datasets`

### Aim
To perform simple data wrangling on housing-price data.

### Tasks
1. Load the dataset.
2. Clean column names.
3. Handle missing values.
4. Merge an additional small dataset.
5. Filter the data.
6. Encode categorical data.
7. Calculate summary statistics.
8. Handle simple outliers.


In [ ]:
import pandas as pd
import numpy as np

from sklearn.datasets import fetch_california_housing

pd.set_option("display.max_columns", None)


## 1. Load the dataset

The **California Housing dataset** is available through `sklearn.datasets`.


In [ ]:
# Load the California Housing dataset
housing = fetch_california_housing(as_frame=True)

df = housing.frame.copy()

print("Dataset shape:", df.shape)
display(df.head())


## 2. Clean the column names

In [ ]:
# Make column names easier to use
df.columns = (
    df.columns
    .str.strip()
    .str.replace(" ", "_")
    .str.lower()
)

# Rename the target column
df = df.rename(columns={"medhouseval": "house_price"})

# Actual sklearn column names are kept in lowercase (e.g. averooms, medinc).

print("Clean column names:")
print(df.columns.tolist())


## 3. Check and handle missing values

In [ ]:
print("Missing values before cleaning:")
print(df.isnull().sum())

# Fill missing numerical values with the median
df = df.fillna(df.median(numeric_only=True))

print("\nMissing values after cleaning:")
print(df.isnull().sum())


## 4. Add a simple categorical column

The original dataset mainly contains numerical information. We create a simple **property type** column so that categorical-data handling can be demonstrated.


In [ ]:
# Create a simple property type
# If a house has more than 5 average rooms, call it "Large".
df["property_type"] = np.where(
    df["averooms"] > 5,
    "Large",
    "Small"
)

display(df[["averooms", "property_type"]].head())


## 5. Merge additional information

In [ ]:
# Create a small additional dataset based on the income level
# This represents simple neighborhood information.
neighborhood_info = pd.DataFrame({
    "income_group": ["Low", "Medium", "High"],
    "amenity_score": [4, 7, 9]
})

# Create the same income group in the housing data
df["income_group"] = pd.cut(
    df["medinc"],
    bins=[0, 2.5, 5, np.inf],
    labels=["Low", "Medium", "High"]
)

# Merge the two datasets
df = df.merge(
    neighborhood_info,
    on="income_group",
    how="left"
)

display(df.head())


## 6. Filter the data

In [ ]:
# Select houses with:
# - more than 5 rooms
# - house price greater than 2
filtered_df = df[
    (df["averooms"] > 5) &
    (df["house_price"] > 2)
]

print("Filtered data:")
display(filtered_df.head())

print("Number of filtered records:", len(filtered_df))


## 7. Encode categorical variables

In [ ]:
# One-hot encode the property_type column
df_encoded = pd.get_dummies(
    df,
    columns=["property_type"],
    drop_first=True
)

display(df_encoded.head())


## 8. Calculate summary statistics

In [ ]:
# Average house price by property type
average_price = (
    df.groupby("property_type")["house_price"]
    .mean()
)

print("Average house price by property type:")
print(average_price)

print("\nOverall summary:")
display(
    df[["medinc", "averooms", "house_price"]].describe()
)


## 9. Handle outliers

In [ ]:
# Simple outlier handling using the IQR method
Q1 = df["house_price"].quantile(0.25)
Q3 = df["house_price"].quantile(0.75)

IQR = Q3 - Q1

lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

# Remove values outside the limits
clean_df = df[
    (df["house_price"] >= lower_limit) &
    (df["house_price"] <= upper_limit)
].copy()

print("Rows before removing outliers:", len(df))
print("Rows after removing outliers :", len(clean_df))


## 10. Final cleaned dataset

In [ ]:
print("Final dataset shape:", clean_df.shape)
display(clean_df.head())

# Save the cleaned dataset
clean_df.to_csv("Cleaned_RealEstate_Prices.csv", index=False)

print("Cleaned dataset exported successfully.")


## Conclusion

The real estate data was wrangled by:
- Loading the California Housing dataset from sklearn
- Cleaning column names
- Handling missing values
- Adding and merging neighborhood information
- Filtering records
- Encoding a categorical variable
- Calculating average house prices
- Detecting and removing simple outliers
- Exporting the cleaned dataset
